# 01. Знакомство с данными и чистка

В этом ноутбуке надо выполнить три простых задачи: понять, что вообще лежит в датасете, найти и убрать мусор, посмотреть базовые срезы по выручке и сезонности. Дальше уже будем считать retention и проверять гипотезы.

Если коротко, то после чистки в данных остаётся около 800 тысяч строк и порядка 5,8 тысяч уникальных клиентов. Этого с запасом хватит для когортного анализа и для статистики.

## Импорты

Подключаю pandas для работы с таблицами, matplotlib и seaborn для графиков. Ничего экзотического.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 50)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

## Загрузка

В исходном Excel-файле два листа, по одному на каждый год наблюдений. Склеиваю их в один датафрейм, чтобы дальше работать с целыми двумя годами без оглядки на источник.

In [ ]:
DATA_PATH = '../data/online_retail_II.xlsx'

df_09_10 = pd.read_excel(DATA_PATH, sheet_name='Year 2009-2010')
df_10_11 = pd.read_excel(DATA_PATH, sheet_name='Year 2010-2011')

df = pd.concat([df_09_10, df_10_11], ignore_index=True)
print(f'Размер: {df.shape[0]:,} строк, {df.shape[1]} колонок')
df.head()

Что в данных:
- Invoice: номер чека. Если начинается с буквы C, это возврат, его потом надо будет убрать.
- StockCode: артикул товара.
- Description: название.
- Quantity: количество. Бывает отрицательным у возвратов.
- InvoiceDate: дата и время покупки.
- Price: цена за единицу.
- Customer ID: идентификатор клиента. Это главное поле для нас, потому что без него мы не можем отслеживать клиента во времени.
- Country: страна доставки.

## Где у нас дыры

Первое, что хочется проверить в любом датасете, это где пропуски. Особенно интересует Customer ID, потому что без него человек у нас анонимный и в когортный анализ просто не попадёт.

In [ ]:
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({'missing': missing, 'pct': missing_pct})

Около 22% строк без Customer ID. Это либо гостевые покупки, либо технические записи. Для retention-анализа эти строки бесполезны, и я их уберу. Сразу зафиксирую потерю, чтобы было ясно, насколько мы порезали выборку.

## Чистка

Что и зачем убираю:
1. Строки без Customer ID. Без идентификатора клиента нет когортного анализа.
2. Возвраты. Чек, начинающийся с буквы C, это отмена, а не покупка.
3. Строки с неположительным количеством или ценой. Это либо ошибки ввода, либо технические проводки.
4. Сразу же добавляю колонку Revenue (выручка по строке) и InvoiceMonth (месяц покупки), они понадобятся в каждом следующем разделе.

In [ ]:
before = len(df)

df = df.dropna(subset=['Customer ID'])
df = df[~df['Invoice'].astype(str).str.startswith('C')]
df = df[(df['Quantity'] > 0) & (df['Price'] > 0)]
df['Customer ID'] = df['Customer ID'].astype(int)
df['Revenue'] = df['Quantity'] * df['Price']
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['InvoiceMonth'] = df['InvoiceDate'].dt.to_period('M').dt.to_timestamp()

print(f'Было строк: {before:,}')
print(f'Стало строк: {len(df):,}')
print(f'Потеряли:   {before - len(df):,} ({(1 - len(df)/before)*100:.1f}%)')

Потеряли четверть, в основном из-за отсутствующих Customer ID. Для розничного датасета это ожидаемо и не искажает дальнейший анализ: оставшиеся строки уже принадлежат клиентам, которых можно отслеживать.

## Базовая статистика

Просто чтобы понимать масштаб того, с чем работаем.

In [ ]:
print(f'Период:           {df["InvoiceDate"].min():%Y-%m-%d}  ->  {df["InvoiceDate"].max():%Y-%m-%d}')
print(f'Уникальных клиентов: {df["Customer ID"].nunique():,}')
print(f'Уникальных заказов:  {df["Invoice"].nunique():,}')
print(f'Уникальных товаров:  {df["StockCode"].nunique():,}')
print(f'Общая выручка:       £{df["Revenue"].sum():,.0f}')
print(f'Стран в данных:      {df["Country"].nunique()}')

## Откуда клиенты

Распределение по странам это короткий, но важный срез. Подозреваю, что доминирует Великобритания, и от этого будут зависеть оговорки в выводах.

In [ ]:
country_revenue = df.groupby('Country')['Revenue'].sum().sort_values(ascending=False).head(10)
ax = country_revenue.plot(kind='barh', figsize=(10, 5), color='#4C72B0')
ax.invert_yaxis()
ax.set_title('Топ-10 стран по выручке')
ax.set_xlabel('Выручка, £')
plt.tight_layout()
plt.show()

Великобритания даёт более 85% выручки, остальные страны это шумный хвост. Дальше анализ можно делать как по всему датасету, так и только по UK, я оставлю всё. В выводах надо помнить, что это локальная история, и переносить цифры один-в-один на другие рынки нельзя.

## Сезонность

Если в декабре всплеск, это надо будет учитывать в когортном анализе: декабрьские покупатели часто приходят за подарками и не возвращаются.

In [ ]:
monthly = df.groupby('InvoiceMonth').agg(
    revenue=('Revenue', 'sum'),
    orders=('Invoice', 'nunique'),
    customers=('Customer ID', 'nunique')
).reset_index()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(monthly['InvoiceMonth'], monthly['revenue'], marker='o', color='#4C72B0')
ax.set_title('Выручка по месяцам')
ax.set_ylabel('Выручка, £')
ax.set_xlabel('Месяц')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Видно ярко выраженный пик в ноябре-декабре обоих лет, это предновогодний шопинг. Картинка типичная для розницы. Для нас это сигнал заранее присмотреться к декабрьским когортам в ноутбуке 02.

## Распределение размера чека

Соберём чеки и посмотрим, как они распределены. Это пригодится в следующем ноутбуке, когда будем сравнивать размер первого чека у вернувшихся и однократных клиентов.

In [ ]:
invoice_totals = df.groupby('Invoice')['Revenue'].sum()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(invoice_totals, bins=80, color='#4C72B0')
axes[0].set_title('Размер чека (как есть, с выбросами)')
axes[0].set_xlabel('£')
axes[0].set_ylabel('Кол-во чеков')

p99 = invoice_totals.quantile(0.99)
axes[1].hist(invoice_totals[invoice_totals < p99], bins=80, color='#55A868')
axes[1].set_title(f'То же, без верхнего 1% (< £{p99:.0f})')
axes[1].set_xlabel('£')
plt.tight_layout()
plt.show()

print(f'Медианный чек: £{invoice_totals.median():.2f}')
print(f'Средний чек:   £{invoice_totals.mean():.2f}')

Распределение сильно скошенное вправо: большинство чеков мелкие, а длинный хвост из крупных чеков (это оптовики). Среднее почти вдвое больше медианы, это и есть классический признак тяжёлого хвоста.

Что это значит для дальнейшего анализа: при сравнении средних надо либо использовать робастные методы, либо открыто признавать, что среднее чувствительно к выбросам, и приводить медианы для перепроверки. Я выбираю t-тест Уэлча и параллельно показываю медианы.

## Сохраняем чистые данные

Чтобы в следующих ноутбуках не повторять чистку, сохраняю результат в parquet. Это быстрее, чем CSV или Excel, и сохраняет типы.

In [ ]:
df.to_parquet('../data/clean.parquet', index=False)
print('Сохранено: data/clean.parquet')

## Что важно унести из этого ноутбука

Данные двухлетние, с заметной декабрьской сезонностью. После чистки осталось 800 тысяч строк и около 5,8 тысяч клиентов, выборки достаточно для статистики. Распределение чеков скошенное, и это будет влиять на выбор статистических методов. Большинство выручки приходит из UK, поэтому выводы локальные.

Дальше переходим к ядру: считаем когортный retention и проверяем, действительно ли размер первого чека отличает вернувшихся от однократных.